## Setup

In [1]:
pip install torch torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 790.4 kB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 5.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 KB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 322.9 kB/s eta 0:00:0000:0100:01�━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.9/170.1 MB 1.2 MB/s eta 0:01:59��━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/170.1 MB 1.2 MB/s eta 0:02:00━━━━━━━━━━━━━━━━━━━━━━ 27.8/170.1 MB 1.1 MB/s eta 0:02:07
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 KB 1.1 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 4.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 4.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 KB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 2.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━

In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

df_train = pd.read_csv('../dataset/processed/train_final_index.csv')
df_val = pd.read_csv('../dataset/processed/val_index.csv')
df_test = pd.read_csv('../dataset/processed/test_index_clean.csv')

EMOTIONS = sorted(df_train['emotion'].unique())
label2idx = {emotion: idx for idx, emotion in enumerate(EMOTIONS)}
print(label2idx)

{'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


carichiamo le immagini in un array numpy

In [3]:
def load_images_to_array(df, size=48):
    arr = np.zeros((len(df), size, size), dtype=np.uint8)
    for i, path in enumerate(df['path']):
        arr[i] = np.array(Image.open(path))
    labels = df['emotion'].map(label2idx).values.astype(np.int64)
    return arr, labels

X_train, y_train = load_images_to_array(df_train)
X_val, y_val = load_images_to_array(df_val)
X_test, y_test = load_images_to_array(df_test)

print(X_train.shape, X_val.shape, X_test.shape)

FileNotFoundError: [Errno 2] No such file or directory: '..\\dataset\\raw\\train\\sad\\Training_66485665.jpg'

Normalizzazione (su train) usanzo z-score

In [4]:
# z-score: media e std calcolate SOLO sul train, riusate identiche su val/test
mean = X_train.mean() / 255.0
std = X_train.std() / 255.0
print(f"mean={mean:.4f}, std={std:.4f}")

NameError: name 'X_train' is not defined

Salviamo gli array precomputati

In [5]:
os.makedirs('../dataset/processed', exist_ok=True)

np.savez('../dataset/processed/fer_arrays.npz',
         X_train=X_train, y_train=y_train,
         X_val=X_val, y_val=y_val,
         X_test=X_test, y_test=y_test,
         mean=mean, std=std)

print("Salvato fer_arrays.npz")

Salvato fer_arrays.npz


Normalizzazione e Agumentation


In [6]:
class FERDataset(Dataset):
    def __init__(self, images, labels, mean, std, train=False, disgust_idx=None):
        self.images = images
        self.labels = labels
        self.mean = mean
        self.std = std
        self.train = train
        self.disgust_idx = disgust_idx

        self.base_augment = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
        ])

        # augmentation più aggressiva, riservata alla classe minoritaria
        self.strong_augment = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3),
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])
        label = self.labels[idx]

        if self.train:
            if self.disgust_idx is not None and label == self.disgust_idx:
                img = self.strong_augment(img)
            else:
                img = self.base_augment(img)

        img_t = torch.from_numpy(np.array(img, dtype=np.float32)) / 255.0
        img_t = (img_t - self.mean) / self.std
        img_t = img_t.unsqueeze(0)  # (1, 48, 48)

        return img_t, torch.tensor(label, dtype=torch.long)

In [7]:
disgust_idx = label2idx['disgust']

train_dataset = FERDataset(X_train, y_train, mean, std, train=True, disgust_idx=disgust_idx)
val_dataset = FERDataset(X_val, y_val, mean, std, train=False)
test_dataset = FERDataset(X_test, y_test, mean, std, train=False)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

# sanity check
imgs, labels = next(iter(train_loader))
print(imgs.shape, imgs.dtype, labels.shape)
print(f"range dopo normalizzazione: [{imgs.min():.2f}, {imgs.max():.2f}]")

torch.Size([64, 1, 48, 48]) torch.float32 torch.Size([64])
range dopo normalizzazione: [-1.98, 1.95]
